# Code 2: Hybrid Pipeline (Cross-Encoder + LLM)
Local model: cross-encoder/nli-distilroberta-base
LLM: Gemini 2.0 Flash (Zero-Shot / Few-Shot / CoT)
Dataset: MultiNLI dev_matched

Routes samples through the Cross-Encoder first, falling back to Gemini 2.0 Flash for low-confidence cases. 
Tests Zero-Shot, Few-Shot, and CoT as fallback strategies.

Experiments:
  A) Cross-Encoder Baseline
  B) Hybrid: CE + Zero-Shot
  C) Hybrid: CE + Few-Shot
  D) Hybrid: CE + CoT


In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder
from google import genai
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import defaultdict

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

# ============================================================
# Configuration
# ============================================================
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Force CPU for compatibility
os.environ["GOOGLE_API_KEY"] = "Your API Key Here"  # Set your API key in environment variable

CE_MODEL_NAME = "cross-encoder/nli-distilroberta-base"
LLM_MODEL = "gemini-2.0-flash"
DATA_FILE = "multinli_1.0_dev_matched.jsonl"
TEST_SIZE = 10          # Change to 2000 for final run
FALLBACK_THRESHOLD = 0.85  # Justified by threshold analysis (see Code 4)
MAX_RETRIES = 6
REQUEST_DELAY = 4
VALID_LABELS = ["contradiction", "entailment", "neutral"]

# Gemini 2.0 Flash pricing (USD per 1M tokens)
COST_INPUT = 0.1
COST_OUTPUT = 0.4

# ============================================================
# Data Loading
# ============================================================
def load_data(path, n):
    samples = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if len(samples) >= n:
                break
            try:
                obj = json.loads(line.strip())
                if obj.get("gold_label") in VALID_LABELS:
                    samples.append(obj)
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(samples)} samples.")
    return samples

# ============================================================
# Prompt Strategies (identical to Code 1 for fair comparison)
# ============================================================
def build_zero_shot(premise, hypothesis):
    return (
        "You are an NLI classifier. Determine if the hypothesis is an "
        "'entailment', 'contradiction', or 'neutral' to the premise.\n"
        "- entailment: the hypothesis MUST be true given the premise.\n"
        "- contradiction: the hypothesis CANNOT be true given the premise.\n"
        "- neutral: the hypothesis MIGHT be true, but the premise doesn't "
        "give enough info to be sure.\n\n"
        "Respond with EXACTLY ONE word. No punctuation. No explanation.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n\n"
        "Label:"
    )

def build_few_shot(premise, hypothesis):
    examples = (
        "Example 1:\n"
        "Premise: \"The program has been in place since 1994 and has helped "
        "thousands of low-income families.\"\n"
        "Hypothesis: \"A program has provided assistance to families in need.\"\n"
        "Label: entailment\n\n"
        "Example 2:\n"
        "Premise: \"He turned around and settled into his chair, looking out "
        "the window at the garden.\"\n"
        "Hypothesis: \"He was thinking about his childhood when he looked "
        "out the window.\"\n"
        "Label: neutral\n\n"
        "Example 3:\n"
        "Premise: \"The new rights are nice enough, but they do not like to "
        "appear to be asked.\"\n"
        "Hypothesis: \"Everyone has always appreciated being asked about "
        "new rights.\"\n"
        "Label: contradiction\n"
    )
    return (
        "You are an expert NLI classifier.\n"
        "Classify the relationship between premise and hypothesis as: "
        "entailment, contradiction, or neutral.\n\n"
        f"{examples}\n"
        "Classify the following pair. Respond with EXACTLY ONE word.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n"
        "Label:"
    )

def build_cot(premise, hypothesis):
    return (
        "You are a strict-logic NLI classifier.\n\n"
        "Use this 3-step verification:\n"
        "Step 1 (Entailment?): Is every claim in the hypothesis logically "
        "supported by the premise without adding new information? "
        "If yes -> entailment.\n"
        "Step 2 (Contradiction?): Does the premise make the hypothesis "
        "impossible? If yes -> contradiction.\n"
        "Step 3 (Neutral?): If the hypothesis adds ANY new details not "
        "proven by the premise, it MUST be neutral.\n\n"
        f"Premise: \"{premise}\"\n"
        f"Hypothesis: \"{hypothesis}\"\n\n"
        "Walk through the 3 steps in 1-2 sentences, then give your label.\n"
        "Reasoning:"
    )

# ============================================================
# Cross-Encoder Inference
# ============================================================
def run_cross_encoder(samples):
    print("Loading Cross-Encoder model...")
    ce = CrossEncoder(CE_MODEL_NAME, device="cpu")
    pairs = [(s["sentence1"], s["sentence2"]) for s in samples]

    print(f"Running inference on {len(pairs)} samples...")
    scores = ce.predict(pairs, show_progress_bar=True)

    preds, confs = [], []
    for score in scores:
        probs = np.exp(score - np.max(score))
        probs = probs / probs.sum()
        preds.append(VALID_LABELS[np.argmax(probs)])
        confs.append(np.max(probs))

    return np.array(preds), np.array(confs)

# ============================================================
# LLM API Call
# ============================================================
def call_llm(client, prompt, max_tokens=10):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL, contents=prompt,
                config={"temperature": 0, "max_output_tokens": max_tokens}
            )
            raw = response.text.strip() if response.text else ""
            in_tok = getattr(response.usage_metadata, "prompt_token_count", 0) or 0
            out_tok = getattr(response.usage_metadata, "candidates_token_count", 0) or 0
            return raw, in_tok, out_tok
        except Exception as e:
            wait = min(2 ** attempt, 60)
            print(f"  API error (attempt {attempt}): {e}, retrying in {wait}s")
            time.sleep(wait)
    return "error", 0, 0

def parse_label(raw, is_cot=False):
    if not raw or "error" in raw:
        return "error"
    text = raw.strip().lower()
    matches = re.findall(r"\b(entailment|contradiction|neutral)\b", text)
    if matches:
        return matches[-1] if is_cot else matches[0]
    return "unknown"

# ============================================================
# Error Analysis
# ============================================================
def print_error_analysis(samples, gold, pred, name, top_n=10):
    print(f"\n--- Error Analysis: {name} ---")
    error_types = defaultdict(int)
    errors = []
    for i, (g, p) in enumerate(zip(gold, pred)):
        if g != p:
            error_types[f"{g} -> {p}"] += 1
            errors.append((i, g, p))

    print(f"Total errors: {len(errors)} / {len(gold)}")
    print("Error type distribution:")
    for t, c in sorted(error_types.items(), key=lambda x: -x[1]):
        print(f"  {t}: {c}")

    print(f"\nTop {min(top_n, len(errors))} error samples:")
    for idx, g, p in errors[:top_n]:
        print(f"  [{idx}] {g} -> {p} | "
              f"P: {samples[idx]['sentence1'][:60]}... | "
              f"H: {samples[idx]['sentence2'][:60]}...")

# ============================================================
# Main
# ============================================================
def main():
    samples = load_data(DATA_FILE, TEST_SIZE)
    y_test = np.array([s["gold_label"] for s in samples])

    # --- Step 1: Cross-Encoder Baseline ---
    ce_preds, ce_confs = run_cross_encoder(samples)

    print(f"\n{'='*60}")
    print("BASELINE: Cross-Encoder Only (nli-distilroberta-base)")
    print(f"{'='*60}")
    ce_acc = accuracy_score(y_test, ce_preds)
    print(f"Accuracy: {ce_acc:.4f}")
    print(classification_report(y_test, ce_preds, labels=VALID_LABELS, zero_division=0))
    print_error_analysis(samples, y_test, ce_preds, "Cross-Encoder Only")

    print(f"\nConfidence distribution:")
    print(f"  Mean: {ce_confs.mean():.4f}, Median: {np.median(ce_confs):.4f}")
    for t in [0.70, 0.80, 0.85, 0.90, 0.95]:
        n = (ce_confs >= t).sum()
        print(f"  >= {t}: {n}/{len(ce_confs)} ({n/len(ce_confs)*100:.1f}%)")

    # --- Step 2: Run LLM on ALL samples (for threshold sweep) ---
    client = genai.Client(api_key=os.environ.get("GOOGLE_API_KEY"))

    strategies = {
        "Zero-Shot": (build_zero_shot, False, 10),
        "Few-Shot":  (build_few_shot, False, 10),
        "CoT":       (build_cot, True, 150),
    }

    llm_results = {}
    for name, (builder, is_cot, max_tok) in strategies.items():
        print(f"\nRunning LLM ({name}) on fallback samples...")

        # Only call LLM for low-confidence samples (save API calls)
        fallback_mask = ce_confs < FALLBACK_THRESHOLD
        fallback_idx = np.where(fallback_mask)[0]
        print(f"  Fallback samples: {len(fallback_idx)}")

        llm_preds = ce_preds.copy()  # Start with CE predictions
        total_in, total_out = 0, 0

        for idx in tqdm(fallback_idx, desc=name):
            s = samples[idx]
            prompt = builder(s["sentence1"], s["sentence2"])
            raw, in_tok, out_tok = call_llm(client, prompt, max_tok)
            llm_preds[idx] = parse_label(raw, is_cot)
            total_in += in_tok
            total_out += out_tok
            time.sleep(REQUEST_DELAY)

        cost = (total_in / 1e6) * COST_INPUT + (total_out / 1e6) * COST_OUTPUT
        llm_results[name] = {
            "preds": llm_preds,
            "in_tok": total_in, "out_tok": total_out, "cost": cost
        }

    # --- Step 3: Final Hybrid Results ---
    print(f"\n{'='*60}")
    print(f"HYBRID RESULTS (threshold = {FALLBACK_THRESHOLD})")
    print(f"{'='*60}")

    rows = []
    for name, r in llm_results.items():
        acc = accuracy_score(y_test, r["preds"])
        rows.append({
            "Strategy": f"Hybrid + {name}",
            "Accuracy": acc,
            "API Calls": int((ce_confs < FALLBACK_THRESHOLD).sum()),
            "Input Tok": r["in_tok"],
            "Output Tok": r["out_tok"],
            "Cost ($)": r["cost"],
        })

        # Per-strategy detailed report
        print(f"\n--- Hybrid + {name}: Accuracy = {acc:.4f} ---")
        print(classification_report(
            y_test, r["preds"], labels=VALID_LABELS, zero_division=0
        ))
        print_error_analysis(samples, y_test, r["preds"], f"Hybrid + {name}")

    # Summary comparison
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")

    fallback_n = int((ce_confs < FALLBACK_THRESHOLD).sum())
    ce_n = len(samples) - fallback_n
    print(f"Routing: CE handled {ce_n}/{len(samples)} "
          f"({ce_n/len(samples)*100:.0f}%), "
          f"LLM fallback {fallback_n} ({fallback_n/len(samples)*100:.0f}%)\n")

    summary_rows = [{"Strategy": "CE Only", "Accuracy": ce_acc,
                     "API Calls": 0, "Input Tok": 0, "Output Tok": 0,
                     "Cost ($)": 0.0}]
    summary_rows.extend(rows)
    print(pd.DataFrame(summary_rows).to_string(
        index=False, float_format="{:.4f}".format))

if __name__ == "__main__":
    main()
